###Deltalake & Lakehouse Optimization Usecases

![](/Workspace/Users/agalya.dotnet.2321@gmail.com/databricks-code-repo_latest/databricks_tasks_2025/5_all_databricks_workouts/DELTA OPTIMIZATIONS.png)


####**_Note_**: For this use case, Silver layer tables are being used because the Gold layer tables were not created successfully in the workspace due to resource not found issues.

####1. Handling Data Skew & Query Performance (Optimize & Z-Order)
Scenario: The analytics team reports that queries filtering silver_shipments by source_city and shipment_date are becoming slow as data volume grows.

Task: Run the OPTIMIZE command with ZORDER on the silver_shipments table to co-locate related data in the same files.

Outcome:
Why did we choose source_city and shipment_date for Z-Ordering instead of shipment_id? Think about high cardinality vs. query filtering

In [0]:
%sql
OPTIMIZE prodcatalog.logistics.silver_shipments 
ZORDER BY(source_city, shipment_date);

Why source_city and shipment_date instead of shipment_id for Z-Ordering?<BR>
**Z-Ordering is most effective when you use columns that are frequently used in query filters and have low to medium cardinality.<BR>
**We avoided shipment_id because its high cardinality provides no clustering benefit. We chose source_city and shipment_date because they align with common query filters and enable effective data skipping, which is exactly what Z-Ordering is designed for.

#### 2. Speeding up Regional Queries (Partition Pruning)
Scenario: The dashboard team reports that queries filtering for orgin_hub_city with "New York" shipments from the gold_core_curated_tbl table are scanning the entire dataset (Terabytes of data), even though New York is only 5% of the data. This is racking up compute costs.

Task: Re-create the gold_core_curated_tbl table partitioned by orgin_hub_city. Run a query filtering for one city to demonstrate "Partition Pruning" (where Spark skips files that don't match the filter).

Outcome: Verify the partition filtering is applied or not, by performing explain plan, check for the PartitionFilters in the output.

In [0]:
%sql

CREATE OR REPLACE TABLE prodcatalog.logistics.silver_shipments_partition
USING DELTA
PARTITIONED BY (source_city )
AS
SELECT *
FROM prodcatalog.logistics.silver_shipments;

In [0]:
%sql
SHOW PARTITIONS prodcatalog.logistics.silver_shipments_partition;

In [0]:
%sql
select * from prodcatalog.logistics.silver_shipments_partition where source_city = 'Chennai'

#### 3. Storage Cost Savings (Vacuum)
Scenario: Your Project pipeline runs every hour, creating many small files and obsolete versions of data. Your storage costs are rising. You need to clean up files that are no longer needed for time travel.

Task: Execute a Vacuum command to remove data files older than the retention threshold.

Outcome: Perform the describe history and find whether vacuum is completed.

In [0]:
%sql
VACUUM prodcatalog.logistics.silver_shipments_partition retain 168 hours

####4. Modern Data Layout (Liquid Clustering)
Scenario: You are redesigning the silver_shipments table. You want to avoid the "small files" problem and need a flexible layout that adapts to changing query patterns automatically without rewriting the table.

Task: Re-create the silver_shipments table using Liquid Clustering on the shipment_id column.

Outcome: Liquid Clustering over traditional partitioning when the cardinality of shipment_id is very high.

In [0]:
%sql
CREATE OR REPLACE TABLE prodcatalog.logistics.silver_shipments_lqd
USING DELTA
CLUSTER BY (shipment_id)
AS
SELECT *
FROM prodcatalog.logistics.silver_shipments;

In [0]:
%sql
 select * from prodcatalog.logistics.silver_shipments_lqd

In [0]:
%sql
describe detail  prodcatalog.logistics.silver_shipments_lqd;

In [0]:
%sql
describe history   prodcatalog.logistics.silver_shipments_lqd;

Handles high cardinality well
Liquid Clustering is built for columns like shipment_id with millions of unique values, where partitioning or Z-ORDERing would be ineffective.

No fixed partitions, less maintenance
It automatically reorganizes data as it grows—no need to manage partitions or keep re-running manual strategies.

Faster queries through smart data skipping
Data gets colocated based on real query patterns, so filters and joins read fewer files and run faster.

#### 5. Cost Efficient Environment Cloning (Shallow Clone)
Scenario: The QA team needs to test an update on the gold_core_curated_tbl table. The table is 5TB in size. You cannot afford to duplicate the storage cost just for a test and the update should not affect the copied table.

Task: Create a Shallow Clone of the gold table for the QA team.

Outcome: If we delete records from the source table (gold_core_curated_tbl), will the QA table (gold_core_curated_tbl_qa) be affected? Why or why not?

In [0]:
%sql
--SHALLOW CLONE
CREATE TABLE prodcatalog.logistics.silver_shipments_shallow_clone
SHALLOW CLONE prodcatalog.logistics.silver_shipments;


Outcome: If we delete records from the source table (silver_shipments), will the QA table (silver_shipments_shallow_clone) be affected? Why or why not? <BR>
**Ans:**  :Yes , In shallow it copies only the metadata , data will get pointed to the source only .Hence the table we created will get affected 

In [0]:
%sql
select * from prodcatalog.logistics.silver_shipments_shallow_clone limit 20;

#### 6. Disaster Recovery (Time Travel & Restore)
Scenario: A junior data engineer accidentally ran a logic error that corrupted the gold_core_curated_tbl table 15 minutes ago. You need to revert the table to its previous state immediately.

Task: Use Delta Lake's Restore feature to roll back the table.

Outcome:What is the difference between querying with VERSION AS OF (Time Travel) and running RESTORE?

In [0]:
%sql
RESTORE TABLE prodcatalog.logistics.silver_shipments_partition
TO VERSION AS OF 1;

Outcome:What is the difference between querying with VERSION AS OF (Time Travel) and running RESTORE?<Br>
**ANS** :<br>
Time Travel lets you visit the past. RESTORE moves the present back there.<BR>
**Time Travel**:<BR>
1)Read-only operation
2)Lets you query past versions of a table<BR>
**Restore**:<BR>
1)Write operation
2)Physically reverts the table to a previous version